<a href="https://colab.research.google.com/github/arisdavidpinogarcia-coder/Arkadoybot-/blob/main/Arkaboy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import time
import logging
from datetime import datetime
from collections import defaultdict

# ==============================
# CONFIGURACIÓN
# ==============================

API_KEY        = "7dd869aa6dc74002b1011c74f05a9b8c"
TELEGRAM_TOKEN = "8675989474:AAGSehhlV4kfhwogvBXqY5eQKHkqTxfXZgc"
CHAT_ID        = "5556309003"

HEADERS = {"x-apisports-key": API_KEY}

# Ligas válidas (IDs de API-Sports)
LIGAS_VALIDAS = {
    39,   # Premier League
    140,  # La Liga
    78,   # Bundesliga
    135,  # Serie A
    61,   # Ligue 1
    94,   # Primeira Liga
    2,    # Champions League
    3,    # Europa League
    71,   # Brasileirao
    262,  # Liga MX
    253,  # MLS
    239,  # Primera A Colombia
}

# Control de rate limit: máximo llamadas por ciclo
MAX_STATS_CALLS_POR_CICLO = 15

# Umbral para alertar (score mínimo)
SCORE_ALERTA  = 8
# Diferencia mínima de score para re-alertar el mismo partido
SCORE_DELTA   = 2

INTERVALO_LOOP = 30  # segundos entre ciclos

# ==============================
# LOGGING
# ==============================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("bot_goles.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger("BotGoles")

# ==============================
# ESTADO GLOBAL
# ==============================

# fixture_id -> último score con el que se alertó (0 = nunca alertado)
alertados: dict[int, int] = {}

# fixture_id -> stats del ciclo anterior (para momentum)
historial: dict[int, dict] = {}

# Contador de requests a la API en el ciclo actual
stats_calls_ciclo = 0

# ==============================
# TELEGRAM
# ==============================

def enviar_telegram(msg: str) -> bool:
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    try:
        res = requests.post(url, json={
            "chat_id": CHAT_ID,
            "text": msg,
            "parse_mode": "HTML"
        }, timeout=10)
        data = res.json()
        if not data.get("ok"):
            log.error(f"Telegram error: {data}")
            return False
        log.info("Telegram enviado OK")
        return True
    except Exception as e:
        log.error(f"Telegram conexión: {e}")
        return False

# ==============================
# HELPERS
# ==============================

def safe_int(val) -> int:
    """Convierte valores de API que pueden ser string, None, o int."""
    try:
        return int(str(val).replace("%", "").strip())
    except (ValueError, TypeError):
        return 0

def get_minuto(fixture: dict) -> int:
    """Minuto real incluyendo tiempo extra (90+3 = 93)."""
    elapsed = safe_int(fixture["status"].get("elapsed"))
    extra   = safe_int(fixture["status"].get("extra"))
    return elapsed + extra

# ==============================
# API
# ==============================

def get_matches() -> list:
    try:
        url = "https://v3.football.api-sports.io/fixtures?live=all"
        res = requests.get(url, headers=HEADERS, timeout=10)
        return res.json().get("response", [])
    except Exception as e:
        log.error(f"get_matches: {e}")
        return []

def get_stats(fixture_id: int) -> list:
    global stats_calls_ciclo
    try:
        url = f"https://v3.football.api-sports.io/fixtures/statistics?fixture={fixture_id}"
        res = requests.get(url, headers=HEADERS, timeout=10)
        stats_calls_ciclo += 1
        return res.json().get("response", [])
    except Exception as e:
        log.error(f"get_stats {fixture_id}: {e}")
        return []

def get_odds_live(fixture_id: int) -> dict | None:
    """Intenta obtener cuota en vivo de Over 2.5 como contexto de mercado."""
    try:
        url = f"https://v3.football.api-sports.io/odds/live?fixture={fixture_id}"
        res = requests.get(url, headers=HEADERS, timeout=10)
        data = res.json().get("response", [])
        if not data:
            return None
        for bookmaker in data[0].get("bookmakers", []):
            for bet in bookmaker.get("bets", []):
                if bet.get("name") == "Goals Over/Under":
                    for odd in bet.get("values", []):
                        if odd.get("value") == "Over 2.5":
                            return {"over25": float(odd.get("odd", 0))}
        return None
    except Exception as e:
        log.error(f"get_odds_live {fixture_id}: {e}")
        return None

# ==============================
# PARSEO
# ==============================

def parse_stats(raw: list) -> dict | None:
    if not raw or len(raw) < 2:
        return None

    data = {
        "shots_total":    [0, 0],  # Total shots on goal por equipo
        "attacks":        [0, 0],  # Dangerous attacks
        "corners":        [0, 0],  # Corner kicks
        "possession":     [0, 0],  # Ball possession %
        "yellow_cards":   [0, 0],
    }

    tipo_map = {
        "Shots on Goal":       "shots_total",
        "Dangerous Attacks":   "attacks",
        "Corner Kicks":        "corners",
        "Ball Possession":     "possession",
        "Yellow Cards":        "yellow_cards",
    }

    for i, team in enumerate(raw[:2]):
        for s in team.get("statistics", []):
            key = tipo_map.get(s["type"])
            if key:
                data[key][i] = safe_int(s.get("value"))

    return data

# ==============================
# SCORING IA
# ==============================

def calcular_score(match: dict, stats: dict, fixture_id: int) -> tuple[int, list[str]]:
    """
    Retorna (score, razones[]) donde score máx real es 18.
    Se normaliza a /15 en el mensaje para claridad.
    """
    score = 0
    razones = []

    minute = get_minuto(match["fixture"])
    home_g = safe_int(match["goals"].get("home"))
    away_g = safe_int(match["goals"].get("away"))
    diff   = abs(home_g - away_g)

    # ── Totales combinados (mejor para Over) ──────────────────
    total_shots   = stats["shots_total"][0]   + stats["shots_total"][1]
    total_attacks = stats["attacks"][0]       + stats["attacks"][1]
    total_corners = stats["corners"][0]       + stats["corners"][1]

    # ── Dominancia (quién presiona) ───────────────────────────
    team_dom = 0 if stats["attacks"][0] >= stats["attacks"][1] else 1

    # ⏱️ Ventana temporal crítica
    if 75 <= minute <= 88:
        score += 4
        razones.append(f"⏱️ Minuto crítico ({minute}')")
    elif 65 <= minute < 75:
        score += 2
        razones.append(f"⏱️ Minuto activo ({minute}')")

    # ⚖️ Marcador
    if diff == 0:
        score += 3
        razones.append("⚖️ Empate — ambos necesitan gol")
    elif diff == 1:
        score += 2
        razones.append("⚖️ 1 gol de diferencia — perdedor presiona")

    # 🎯 Tiros totales
    if total_shots >= 12:
        score += 3
        razones.append(f"🎯 {total_shots} tiros en puerta (muy alto)")
    elif total_shots >= 7:
        score += 2
        razones.append(f"🎯 {total_shots} tiros en puerta")
    elif total_shots >= 4:
        score += 1
        razones.append(f"🎯 {total_shots} tiros en puerta")

    # 🚨 Presión ofensiva
    if total_attacks >= 80:
        score += 3
        razones.append(f"🚨 {total_attacks} ataques peligrosos (élite)")
    elif total_attacks >= 50:
        score += 2
        razones.append(f"🚨 {total_attacks} ataques peligrosos")
    elif total_attacks >= 30:
        score += 1
        razones.append(f"🚨 {total_attacks} ataques peligrosos")

    # 🚩 Corners
    if total_corners >= 10:
        score += 2
        razones.append(f"🚩 {total_corners} corners (zona peligro)")
    elif total_corners >= 6:
        score += 1
        razones.append(f"🚩 {total_corners} corners")

    # 🧠 Momentum (aceleración de presión)
    if fixture_id in historial:
        prev = historial[fixture_id]
        delta_attacks = total_attacks - (prev["attacks"][0] + prev["attacks"][1])
        delta_shots   = total_shots   - (prev["shots_total"][0] + prev["shots_total"][1])

        if delta_attacks > 15:
            score += 3
            razones.append(f"📈 Momentum fuerte (+{delta_attacks} ataques)")
        elif delta_attacks > 8:
            score += 2
            razones.append(f"📈 Momentum creciente (+{delta_attacks} ataques)")

        if delta_shots > 4:
            score += 1
            razones.append(f"📈 Tiros en aumento (+{delta_shots})")

    # 🟨 Presión de tarjetas (tensión = más fouls = más peligro)
    total_yellows = stats["yellow_cards"][0] + stats["yellow_cards"][1]
    if total_yellows >= 4:
        score += 1
        razones.append(f"🟨 {total_yellows} amarillas — partido caliente")

    # Actualizar historial
    historial[fixture_id] = stats

    return score, razones


# Máximo teórico: 4+3+3+3+2+3+1+1 = 20 → normalizamos display a /15
SCORE_MAXIMO_DISPLAY = 15

def normalizar_score(score: int) -> int:
    return min(score, SCORE_MAXIMO_DISPLAY)

# ==============================
# ESTRATEGIA
# ==============================

def generar_apuesta(score: int, diff: int, minute: int, odds: dict | None) -> str | None:
    odds_str = ""
    if odds and odds.get("over25"):
        cuota = odds["over25"]
        ev_str = f" | Cuota Over 2.5: {cuota:.2f}"
        odds_str = ev_str

    # Partido empatado en tiempo crítico = señal más limpia
    if score >= 13 and diff == 0 and minute >= 75:
        return f"🔥🔥 ENTRADA PREMIUM: Próximo gol / BTTS{odds_str}"
    elif score >= 11:
        return f"🔥 ENTRADA FUERTE: Próximo gol / Over{odds_str}"
    elif score >= SCORE_ALERTA:
        return f"⚡ ENTRADA ESTÁNDAR: Gol en próximos minutos{odds_str}"
    return None

# ==============================
# LIMPIEZA DE MEMORIA
# ==============================

def limpiar_estado(ids_activos: set):
    """Elimina datos de partidos ya terminados para evitar memory leak."""
    for fid in list(historial.keys()):
        if fid not in ids_activos:
            del historial[fid]
    for fid in list(alertados.keys()):
        if fid not in ids_activos:
            del alertados[fid]

# ==============================
# MAIN LOOP
# ==============================

def main():
    global stats_calls_ciclo

    log.info("🚀 BOT PRO INICIADO")
    enviar_telegram("🤖 <b>Bot de goles PRO v2.0 activado</b>\nMonitoreando ligas top en tiempo real.")

    ciclo = 0

    while True:
        ciclo += 1
        stats_calls_ciclo = 0
        log.info(f"── Ciclo #{ciclo} ──────────────────────")

        try:
            matches = get_matches()
            log.info(f"📡 Partidos en vivo: {len(matches)}")

            # Filtrar por liga válida PRIMERO
            matches_validos = [
                m for m in matches
                if m["league"]["id"] in LIGAS_VALIDAS
            ]
            log.info(f"🏟️  En ligas válidas: {len(matches_validos)}")

            ids_activos = {m["fixture"]["id"] for m in matches_validos}
            limpiar_estado(ids_activos)

            for m in matches_validos:

                # Rate limit por ciclo
                if stats_calls_ciclo >= MAX_STATS_CALLS_POR_CICLO:
                    log.warning("⚠️ Límite de llamadas alcanzado en este ciclo")
                    break

                fixture_id = m["fixture"]["id"]
                minute     = get_minuto(m["fixture"])

                if minute < 60:
                    continue

                # Re-alerta solo si el score subió significativamente
                ultimo_score = alertados.get(fixture_id, 0)

                stats_raw = get_stats(fixture_id)
                stats     = parse_stats(stats_raw)

                if not stats:
                    log.warning(f"  Sin stats: {fixture_id}")
                    continue

                score, razones = calcular_score(m, stats, fixture_id)
                score_norm = normalizar_score(score)

                log.info(
                    f"  [{m['teams']['home']['name']} vs {m['teams']['away']['name']}]"
                    f" min={minute} score={score_norm}/{SCORE_MAXIMO_DISPLAY}"
                )

                debe_alertar = (
                    score >= SCORE_ALERTA and
                    score > ultimo_score + SCORE_DELTA
                )

                if not debe_alertar:
                    continue

                home   = m["teams"]["home"]["name"]
                away   = m["teams"]["away"]["name"]
                home_g = safe_int(m["goals"].get("home"))
                away_g = safe_int(m["goals"].get("away"))
                diff   = abs(home_g - away_g)
                liga   = m["league"]["name"]

                odds = get_odds_live(fixture_id)

                apuesta = generar_apuesta(score, diff, minute, odds)
                if not apuesta:
                    continue

                razones_str = "\n".join(f"  • {r}" for r in razones)

                msg = (
                    f"🔥 <b>ALERTA IA — GOL PROBABLE</b>\n\n"
                    f"🏟️ {liga}\n"
                    f"⚽ <b>{home}</b> {home_g} - {away_g} <b>{away}</b>\n"
                    f"⏱️ Minuto: {minute}'\n\n"
                    f"📊 Score IA: <b>{score_norm}/{SCORE_MAXIMO_DISPLAY}</b>\n\n"
                    f"📋 Señales detectadas:\n{razones_str}\n\n"
                    f"💡 <b>{apuesta}</b>"
                )

                if enviar_telegram(msg):
                    alertados[fixture_id] = score
                    log.info(f"  ✅ Alerta enviada. score={score_norm}")

                # Pausa entre calls para no saturar API
                time.sleep(1.5)

        except Exception as e:
            log.error(f"Error en loop principal: {e}", exc_info=True)

        log.info(f"💤 Esperando {INTERVALO_LOOP}s... (stats calls este ciclo: {stats_calls_ciclo})")
        time.sleep(INTERVALO_LOOP)


if __name__ == "__main__":
    main()

KeyboardInterrupt: 